Time-series work in Pandas revolves around **3 pillars:**
1. **Datetime dtype (`datetime64[ns]`)**
2. **DatetimeIndex**
3. **Frequency-aware operations** (resampling, rolling)

### Creating & Converting Datetime Data

In [1]:
import pandas as pd

df = pd.DataFrame({
    "date": [
        "2024-01-01",
        "2024-01-02",
        "2024-01-03",
        "2024-01-05",
        "invalid_date"
    ],
    "sales": [100, 120, 130, 150, 200]
})

df.dtypes

date     object
sales     int64
dtype: object

#### Convert to datetime

In [2]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df

,date,sales
0,2024-01-01,100
1,2024-01-02,120
2,2024-01-03,130
3,2024-01-05,150
4,NaT,200


- Invalid strings -> `NaT`
- No crashes

Always use `errors="coerce"` for external data.

### DatetimeIndex
Many time-series features only work when **date is the index.**

In [3]:
df = df.set_index("date")
df

,sales
date,
2024-01-01,100
2024-01-02,120
2024-01-03,130
2024-01-05,150
NaT,200


Now the following operations can be performed:
- Resampling
- Time slicing
- Rolling windows

### Time-Based Indexing & Slicing

#### Select a specific date

In [4]:
df.loc["2024-01-02"]

sales    120
Name: 2024-01-02 00:00:00, dtype: int64

#### Select by month/year

Key points:
- `"invalid_date"` -> `NaT`
- The index is **not guaranteed to be sorted**
- **Partial string indexing** like `df.loc["2024"]` **requires a sorted DateTimeIndex**

In [5]:
df = df.sort_index()

In [6]:
df.loc["2024"]

,sales
date,
2024-01-01,100
2024-01-02,120
2024-01-03,130
2024-01-05,150


In [7]:
df.loc["2024-01"]

,sales
date,
2024-01-01,100
2024-01-02,120
2024-01-03,130
2024-01-05,150


### Extracting Datetime Features (`.dt` accessor)

`.dt` is a **vectorized datetime accessor.**

That means:
- It works on an **entire Series at once**
- It exposes **datetime properties and methods**
- It is **fast, safe,** and **NumPy-backed**

Think of `.dt` as:
> "The datetime equivalent of `.str` for strings”

`.dt` works **ONLY IF** the Series dtype is:
- `datetime64[ns]`
- `timedelta64[ns]`
- `Period`

It does **NOT** work on:
- strings
- object dtype
- Python `datetime` objects inside `object`

#### Column vs Index
*If date is a column*    
Use `.dt`:
```python
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
```

*If date is the index (DatetimeIndex)*   
DO NOT use `.dt`:
```python
df.index.year      
df.index.month    
```

In [8]:
df["year"] = df.index.year
df["month"] = df.index.month
df["day"] = df.index.day
df["day_name"] = df.index.day_name()
df["week"] = df.index.isocalendar().week

In [9]:
df

,sales,year,month,day,day_name,week
date,,,,,,
2024-01-01,100,2024.0,1.0,1.0,Monday,1
2024-01-02,120,2024.0,1.0,2.0,Tuesday,1
2024-01-03,130,2024.0,1.0,3.0,Wednesday,1
2024-01-05,150,2024.0,1.0,5.0,Friday,1
NaT,200,NaN,NaN,NaN,NaN,<NA>


### Resampling
Resampling = **GroupBy on time.**

Resampling ONLY works when: 
> The index is a **DatetimeIndex, PeriodIndex,** or **TimedeltaIndex**

In [10]:
monthly_sales = df.resample("M").sum()
monthly_sales

/var/folders/6t/gr1hsb2x571bqm8sycqy4h440000gn/T/ipykernel_1511/3426211089.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_sales = df.resample("M").sum()


,sales,year,month,day,day_name,week
date,,,,,,
2024-01-31,500,8096.0,4.0,11.0,MondayTuesdayWednesdayFriday,4


Common Frequencies:
| Code    | Meaning        |
| ------- | -------------- |
| `D`     | Day            |
| `W`     | Week (Sun end) |
| `W-MON` | Week (Mon end) |
| `M`     | Month end      |
| `MS`    | Month start    |
| `Q`     | Quarter end    |
| `QS`    | Quarter start  |
| `Y`     | Year end       |
| `H`     | Hour           |
| `15min` | 15 minutes     |


In [11]:
df.resample("W").agg({
    "sales": ["sum", "mean", "max"]
})

sales            
             sum   mean  max
date                        
2024-01-07   500  125.0  150

### Handling Missing Dates

#### Reindex to continuous dates

In [12]:
full_range = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="D"
)

df = df.reindex(full_range)

In [13]:
df

,sales,year,month,day,day_name,week
2024-01-01,100.0,2024.0,1.0,1.0,Monday,1
2024-01-02,120.0,2024.0,1.0,2.0,Tuesday,1
2024-01-03,130.0,2024.0,1.0,3.0,Wednesday,1
2024-01-04,NaN,NaN,NaN,NaN,NaN,<NA>
2024-01-05,150.0,2024.0,1.0,5.0,Friday,1


Missing days appear as `NaN`

In [14]:
df["sales"] = df["sales"].ffill()  
df["sales"] = df["sales"].bfill()   

Use only when time continuity makes sense.

### Rolling Windows
Rolling = **windowed computation over time.**

#### 3-day moving average

In [15]:
df["rolling_avg_3"] = df["sales"].rolling(window=3).mean()
df

,sales,year,month,day,day_name,week,rolling_avg_3
2024-01-01,100.0,2024.0,1.0,1.0,Monday,1,NaN
2024-01-02,120.0,2024.0,1.0,2.0,Tuesday,1,NaN
2024-01-03,130.0,2024.0,1.0,3.0,Wednesday,1,116.666667
2024-01-04,130.0,NaN,NaN,NaN,NaN,<NA>,126.666667
2024-01-05,150.0,2024.0,1.0,5.0,Friday,1,136.666667


#### Rolling with minimum data points

In [ ]:
df["rolling_avg_3"] = df["sales"].rolling(
    window=3,
    min_periods=1
).mean()

df

,sales,year,month,day,day_name,week,rolling_avg_3
2024-01-01,100.0,2024.0,1.0,1.0,Monday,1,100.000000
2024-01-02,120.0,2024.0,1.0,2.0,Tuesday,1,110.000000
2024-01-03,130.0,2024.0,1.0,3.0,Wednesday,1,116.666667
2024-01-04,130.0,NaN,NaN,NaN,NaN,<NA>,126.666667
2024-01-05,150.0,2024.0,1.0,5.0,Friday,1,136.666667


### Expanding Windows (Cumulative Metrics)

In [18]:
df["cumulative_sales"] = df["sales"].expanding().sum()

Used for:
- Cumulative revenue
- Growth curves

### Time-Based GroupBy (Advanced Pattern)

#### Daily sales per city
```python
df.groupby([
    pd.Grouper(freq="M"),
    "city"
])["sales"].sum()
```

### Lag Features
#### Previous day sales

In [21]:
df["lag_1"] = df["sales"].shift(1)
df["lag_7"] = df["sales"].shift(7)

df

,sales,year,month,day,day_name,week,rolling_avg_3,cumulative_sales,lag_1,lag_7
2024-01-01,100.0,2024.0,1.0,1.0,Monday,1,100.000000,100.0,NaN,NaN
2024-01-02,120.0,2024.0,1.0,2.0,Tuesday,1,110.000000,220.0,100.0,NaN
2024-01-03,130.0,2024.0,1.0,3.0,Wednesday,1,116.666667,350.0,120.0,NaN
2024-01-04,130.0,NaN,NaN,NaN,NaN,<NA>,126.666667,480.0,130.0,NaN
2024-01-05,150.0,2024.0,1.0,5.0,Friday,1,136.666667,630.0,130.0,NaN


Core concept in:
- Forecasting
- Time-series ML

### Exercise 1
```python
df = pd.DataFrame({
    "date": [
        "2024-01-01",
        "2024-01-02",
        "2024-01-04",
        "2024-01-07",
        "2024-01-08",
        "invalid_date"
    ],
    "city": ["Pune", "Pune", "Mumbai", "Mumbai", "Pune", "Delhi"],
    "sales": [100, 120, 200, 180, 130, 90]
})
```
1. Convert `date` to datetime **safely**
2. Count how many invalid dates exist
3. Drop rows with invalid dates **explicitly**
4. Verify dtype of `date`


In [32]:
import pandas as pd

df = pd.DataFrame({
    "date": [
        "2024-01-01",
        "2024-01-02",
        "2024-01-04",
        "2024-01-07",
        "2024-01-08",
        "invalid_date"
    ],
    "city": ["Pune", "Pune", "Mumbai", "Mumbai", "Pune", "Delhi"],
    "sales": [100, 120, 200, 180, 130, 90]
})

In [33]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df

,date,city,sales
0,2024-01-01,Pune,100
1,2024-01-02,Pune,120
2,2024-01-04,Mumbai,200
3,2024-01-07,Mumbai,180
4,2024-01-08,Pune,130
5,NaT,Delhi,90


In [34]:
df["date"].isna().sum()

np.int64(1)

In [35]:
df = df.dropna(subset=["date"])
df

,date,city,sales
0,2024-01-01,Pune,100
1,2024-01-02,Pune,120
2,2024-01-04,Mumbai,200
3,2024-01-07,Mumbai,180
4,2024-01-08,Pune,130


In [36]:
df["date"].dtype

dtype('<M8[ns]')

### Exercise 2
1. Set `date` as the index
2. Sort the index
3. Verify:
```python
df.index.is_monotonic_increasing
```

In [37]:
df = df.set_index("date")

In [38]:
df

,city,sales
date,,
2024-01-01,Pune,100
2024-01-02,Pune,120
2024-01-04,Mumbai,200
2024-01-07,Mumbai,180
2024-01-08,Pune,130


In [39]:
df = df.sort_index()

In [40]:
df.index.is_monotonic_increasing

True

### Exercise 3
1. Select **all rows from year 2024**
2. Select **January 2024**
3. Try slicing:
```python
df.loc["2024-01-01":"2024-01-03"]
```
4. Observe:
   - Does it work?
   - If not, **why**?

In [41]:
df.loc["2024"]

,city,sales
date,,
2024-01-01,Pune,100
2024-01-02,Pune,120
2024-01-04,Mumbai,200
2024-01-07,Mumbai,180
2024-01-08,Pune,130


In [42]:
df.loc["2024-01"]

,city,sales
date,,
2024-01-01,Pune,100
2024-01-02,Pune,120
2024-01-04,Mumbai,200
2024-01-07,Mumbai,180
2024-01-08,Pune,130


In [44]:
df.loc["2024-01-01":"2024-01-03"]

,city,sales
date,,
2024-01-01,Pune,100
2024-01-02,Pune,120


- `"2024-01-02"` and `"2024-01-03"` **do not exist**
- Pandas refuses value-based slicing on missing timestamps
- This prevents **silent logical bugs**

### Exercise 4
1. Reindex the DataFrame to a **daily frequency**
2. Confirm missing dates now appear as `NaN`
3. Forward-fill `sales`
4. Forward-fill `city`

In [45]:
df = df.asfreq("D")

In [46]:
df

,city,sales
date,,
2024-01-01,Pune,100.0
2024-01-02,Pune,120.0
2024-01-03,NaN,NaN
2024-01-04,Mumbai,200.0
2024-01-05,NaN,NaN
2024-01-06,NaN,NaN
2024-01-07,Mumbai,180.0
2024-01-08,Pune,130.0


In [47]:
df["sales"] = df["sales"].ffill()

In [48]:
df["city"] = df["city"].ffill()

,city,sales
date,,
2024-01-01,Pune,100.0
2024-01-02,Pune,120.0
2024-01-03,Pune,120.0
2024-01-04,Mumbai,200.0
2024-01-05,Mumbai,200.0
2024-01-06,Mumbai,200.0
2024-01-07,Mumbai,180.0
2024-01-08,Pune,130.0


### Exercise 5
1. Compute **daily total sales**
2. Compute **weekly sales sum**
3. Compute **weekly average sales**

In [51]:
daily_sales = df.resample("D").sum(numeric_only=True)
daily_sales

,sales
date,
2024-01-01,100.0
2024-01-02,120.0
2024-01-03,120.0
2024-01-04,200.0
2024-01-05,200.0
2024-01-06,200.0
2024-01-07,180.0
2024-01-08,130.0


In [52]:
weekly_sum = df.resample("W").sum(numeric_only=True)
weekly_sum

,sales
date,
2024-01-07,1120.0
2024-01-14,130.0


In [53]:
weekly_avg = df.resample("W").mean(numeric_only=True)
weekly_avg

,sales
date,
2024-01-07,160.0
2024-01-14,130.0


### Exercise 6
1. Compute a **3-day rolling average of sales**
2. Use `min_periods=1`
3. Explain:
    - Why rolling produces NaNs
    - Why `min_periods` matters

In [54]:
df["rolling_avg_3"] = df["sales"].rolling(
    window=3,
    min_periods=1
).mean()

df

,city,sales,rolling_avg_3
date,,,
2024-01-01,Pune,100.0,100.000000
2024-01-02,Pune,120.0,110.000000
2024-01-03,Pune,120.0,113.333333
2024-01-04,Mumbai,200.0,146.666667
2024-01-05,Mumbai,200.0,173.333333
2024-01-06,Mumbai,200.0,200.000000
2024-01-07,Mumbai,180.0,193.333333
2024-01-08,Pune,130.0,170.000000


Why rolling produces NaNs
- First rows don’t have enough history
- Rolling windows look **backward**

Why min_periods matters   
Without it:
- First 2 rows → `NaN`  

With it:
- Model sees early trend
- Fewer dropped rows

### Exercise 7
1. Create:
   - `lag_1` -> previous day sales
   - `lag_3` -> sales 3 days ago
2. Count how many NaNs each lag produces
3. Explain **why this is expected**

In [55]:
df["lag_1"] = df["sales"].shift(1)
df["lag_3"] = df["sales"].shift(3)

In [56]:
df["lag_1"].isna().sum()  # 1
df["lag_3"].isna().sum()  # 3

np.int64(3)

- Lag uses **past data**
- First `n` rows cannot have `lag_n`
- This is **correct and unavoidable**

### Exercise 8
1. Compute **weekly sales per city**
2. Ensure output has:
   - time index
   - city as a column (not index)

In [58]:
weekly_city_sales = (
    df
    .reset_index()
    .groupby([
        pd.Grouper(key="date", freq="W"),
        "city"
    ])["sales"]
    .sum()
    .reset_index()
)
weekly_city_sales


,date,city,sales
0,2024-01-07,Mumbai,780.0
1,2024-01-07,Pune,340.0
2,2024-01-14,Pune,130.0
